In [40]:
# Get device
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [41]:
# Import and create models
import TailKinematicsNN
import torch

TailKinematicsModel = TailKinematicsNN.TailKinematicsRNN().to(device)
TailKinematicsModel.load_state_dict(torch.load('./Models/kinematicsMoments_0.9991245342.pt'))

# count parameters
total_params = sum(p.numel() for p in TailKinematicsModel.parameters() if p.requires_grad)
print(f'Total trainable parameters in TailKinematicsModel: {total_params}')

Total trainable parameters in TailKinematicsModel: 172


In [42]:
# load Matlab data
import scipy.io
import numpy as np

data = scipy.io.loadmat('./Data/data2025-10-18_00-01-54.mat')
data_keys = ['xout', 'Tail_Forces_out', 'Tail_Centre_out', 'cout', 'cal_stout', 'Moments_Add_out']

# Create a dictionary to hold the data (labels as keys and numpy arrays as values)
data_dict = {key: np.array(data[key]) for key in data_keys}

In [43]:
print(data_dict['cal_stout'].shape)  # Example: print the shape of 'cal_stout' data
data_dict['cal_stout'] = data_dict['cal_stout'][:, :8]  # Keep only the first 8 columns

# Prepare input kinematics tensor (8 motor angular positions + tail centre position))
in_kinematics = np.hstack((data_dict['cal_stout'], data_dict['Tail_Centre_out']))

kinematics_tensor = torch.from_numpy(in_kinematics).float().unsqueeze(-1).to(device)  # Add batch dimension
print(kinematics_tensor.shape)

out_kinematics = np.hstack((data_dict['Moments_Add_out'], data_dict['cout'])) # Target output: moments + caudal amplitude + angle
kinematics_target = torch.from_numpy(out_kinematics).float().to(device)
print(kinematics_target.shape)

(10000, 10)
torch.Size([10000, 9, 1])
torch.Size([10000, 3])


In [44]:
import time
# Get prediction time
start_time = time.time()
TailKinematicsModel.eval()
with torch.no_grad():
    kinematics_preds = TailKinematicsModel(kinematics_tensor)
    print(kinematics_preds.shape)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds.shape[0]
print(f"Prediction total time: {total_time} seconds")
print(f"Mean prediction time: {mean_time_per_prediction} seconds")

torch.Size([10000, 3])
Prediction total time: 0.023197650909423828 seconds
Mean prediction time: 2.3197650909423826e-06 seconds


In [45]:
# Test compilted model
compiled_model = torch.compile(TailKinematicsModel)
start_time = time.time()
compiled_model.eval()
with torch.no_grad():
    kinematics_preds_compiled = compiled_model(kinematics_tensor)
    print(kinematics_preds_compiled.shape)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds_compiled.shape[0]
print(f"Compiled model prediction total time: {total_time} seconds")
print(f"Compiled model mean prediction time: {mean_time_per_prediction} seconds")

torch.Size([10000, 3])
Compiled model prediction total time: 0.003793478012084961 seconds
Compiled model mean prediction time: 3.793478012084961e-07 seconds


In [46]:
# compare single sample prediction times
import time
TailKinematicsModel.eval()
start_time = time.time()
for data in kinematics_tensor:
    with torch.no_grad():
        single_pred = TailKinematicsModel(data.view(1, -1, 1))
end_time = time.time()
print(f"Single sample prediction time: {(end_time - start_time)/kinematics_tensor.shape[0]} seconds")

# Compiled model
start_time = time.time()
for data in kinematics_tensor:
    with torch.no_grad():
        single_pred_compiled = compiled_model(data.unsqueeze(0))
end_time = time.time()
print(f"Single sample prediction time: {(end_time - start_time)/kinematics_tensor.shape[0]} seconds")


Single sample prediction time: 0.0005293974161148071 seconds
Single sample prediction time: 0.0005519329071044922 seconds


In [48]:
import ThrustNN

input_size = 16
hidden_layers =5
npl = 32
output_size = 7
ThrustModel = ThrustNN.thrustFlexNN(input_size=input_size, hidden_size=npl, output_size=output_size, hidden_layers=hidden_layers, dropout_enabled=True).to(device)
ThrustModel.load_state_dict(torch.load('./Models/simple_data_modelv2_32_neurons_5_layers.pt'))

# Compile model
compiled_thrust_model = torch.compile(ThrustModel).to(device)


In [49]:
# Prepare input for Thrust Model with simulation data
#load scalers used during training
import joblib
scaler_in = joblib.load('./Scalers/scaler_thrust_in.pkl')
scaler_out = joblib.load('./Scalers/scaler_thrust_out.pkl')
# Concatenate inputs for Thrust Model
caudal_old = np.hstack((np.zeros(1),data_dict['cout'][:-1, 0])).reshape(-1,1)  # Previous caudal angle (shifted by one timestep)
print(caudal_old.shape)
in_thrust = np.hstack((data_dict['xout'], data_dict['cout'], caudal_old, data_dict['Tail_Centre_out']))  # Input: fluid velocity, caudal angle, previous caudal angle, tail forces
print(in_thrust.shape)
# Scale input data 
in_thrust = scaler_in.transform(in_thrust)  # Scale input data
# Convert to tensor
in_thrust_tensor = torch.from_numpy(in_thrust).float().to(device)

# Prepare target output for Thrust Model
thrust_target = data_dict['Tail_Forces_out']  # Target output: thrust forces
print(thrust_target.shape)
# Exclude all columns that are zero constants
thrust_target = thrust_target[:, ~np.all(thrust_target == 0, axis=0)]
# Scale target data
thrust_target = scaler_out.transform(thrust_target)  # Scale target data
# Convert to tensor
thrust_target_tensor = torch.from_numpy(thrust_target).float().to(device)
print(thrust_target_tensor.shape)

(10000, 1)
(10000, 16)
(10000, 12)
torch.Size([10000, 7])


In [62]:
# Compare normal vs compiled model prediction times
import time
start_time = time.time()
with torch.no_grad():
    TailKinematicsModel.eval()
    kinematics_preds = TailKinematicsModel(kinematics_tensor)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds.shape[0]
print(f"Normal model prediction total time: {total_time} seconds")
print(f"Normal model mean prediction time: {mean_time_per_prediction} seconds")

Normal model prediction total time: 0.002064943313598633 seconds
Normal model mean prediction time: 2.0649433135986328e-07 seconds


In [ ]:
# Compare normal vs compiled model prediction times
import time
start_time = time.time()
with torch.no_grad():
    compiled_model.eval()
    kinematics_preds = compiled_model(kinematics_tensor)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds.shape[0]
print(f"Compiled model prediction total time: {total_time} seconds")
print(f"Compiled model mean prediction time: {mean_time_per_prediction} seconds")

Compiled model prediction total time: 0.0016493797302246094 seconds
Compiled model mean prediction time: 1.6493797302246093e-07 seconds


: 